In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [2]:
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# import pandas_profiling
# train_data.profile_report()

In [4]:
# profile = train_data.profile_report(title='Pandas Profiling Report')
# profile.to_file("Titanic data profiling.html")

In [5]:
train_data.shape

(891, 12)

In [6]:
train_data.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [7]:
miss_val = train_data.isnull().sum().sort_values(ascending=False)
miss_val = pd.DataFrame(data=miss_val, columns=['MissValCount'])
miss_val

,MissValCount
Cabin,687
Age,177
Embarked,2
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
SibSp,0
Parch,0


In [8]:
miss_val['Percent'] = miss_val['MissValCount'].apply(lambda x: '{:.2f}'.format(float(x)/train_data.shape[0]*100))
miss_val_per = miss_val[miss_val['MissValCount'] > 0]
miss_val_per

,MissValCount,Percent
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22


In [9]:
train_data1 = train_data.drop('Cabin', axis=1)
# train_data.dropna(inplace=True)

In [10]:
train_data1.dropna(inplace=True)
train_data1

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S
...,...,...,...,...,...,...,...,...,...,...,...
885,886,0,3,"Rice, Mrs. William (Margaret Norton)",female,39.0,0,5,382652,29.1250,Q
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C


Now we only have 712 rows, with no more missing values. Let's check our test_data now.

In [11]:
test_data = pd.read_csv('/kaggle/input/titanic/test.csv')
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [12]:
miss_val_2 = test_data.isnull().sum().sort_values(ascending=False)
miss_val_2 = pd.DataFrame(miss_val_2, columns=['MissValCount'])
miss_val_2

,MissValCount
Cabin,327
Age,86
Fare,1
PassengerId,0
Pclass,0
Name,0
Sex,0
SibSp,0
Parch,0
Ticket,0


In [13]:
test_data1 = test_data.drop('Cabin', axis=1)
test_data1.dropna(inplace=True)
test_data1.isnull().sum()

PassengerId    0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

In [14]:
women_survived = train_data.loc[(train_data['Sex'] == 'female') & (train_data['Survived'] ==1)]
women = train_data.loc[train_data.Sex == 'female']["Survived"]

In [15]:
rate_women = sum(women)/len(women)*100
print(f'Rate of women who survived: {round(rate_women,2)}%')

Rate of women who survived: 74.2%


In [16]:
men = train_data.loc[train_data['Sex'] == 'male']['Survived']
rate_men = sum(men)/len(men)*100
print(f'Rate of men who survived: {round(rate_men,2)}%')

Rate of men who survived: 18.89%


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

y = train_data['Survived']
features = ['Pclass', 'Sex', 'SibSp', 'Parch']
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test) #y_pred

In [18]:
output = pd.DataFrame({'PassengerId':test_data['PassengerId'],
                      'Survived':predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


In [19]:
X_test.fillna(X_test.mean(), inplace=True)
X_test

,Pclass,SibSp,Parch,Sex_female,Sex_male
0,3,0,0,0,1
1,3,1,0,1,0
2,2,0,0,0,1
3,3,0,0,0,1
4,3,1,1,1,0
...,...,...,...,...,...
413,3,0,0,0,1
414,1,0,0,1,0
415,3,0,0,0,1
416,3,0,0,0,1


In [20]:
# set the new target and predictors

y = train_data1['Survived']  # target

# use only those input features with numeric data type
df_temp = train_data1.drop(['Name', 'Ticket', 'Embarked'], axis=1)

df_temp['Sex'] = df_temp['Sex'].replace('male', 1)
df_temp['Sex'] = df_temp['Sex'].replace('female', 2)

X = df_temp.drop(['Survived'], axis=1) # predictors
X_test = test_data.drop(['Name', 'Ticket', 'Embarked', 'Cabin'], axis=1)
X_test['Sex'] = X_test['Sex'].replace('male', 1)
X_test['Sex'] = X_test['Sex'].replace('female', 2)
X_test.fillna(X_test.mean(), inplace=True)

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test) #y_pred

In [21]:
output = pd.DataFrame({'PassengerId':test_data['PassengerId'],
                      'Survived':predictions})
output.to_csv('new_submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
